In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.pipeline import Pipeline
from scikeras.wrappers import KerasClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pickle


In [5]:
data = pd.read_csv('Churn_Modelling.csv')
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

label_encoder_gender =  LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

OneHotEncoder_geo = OneHotEncoder(handle_unknown='ignore') 
geo_encoded = OneHotEncoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=OneHotEncoder_geo.get_feature_names_out(['Geography']))


data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)

X = data.drop('Exited', axis=1)
y = data['Exited']




In [6]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.transform(X_test)


In [ ]:
#save all the files
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scalar, f)

with open('onehot_encoder_geo.pkl', 'wb') as f:
    pickle.dump(OneHotEncoder_geo, f)
    
with open('label_encoder_gender.pkl', 'wb') as f:
    pickle.dump(label_encoder_gender, f)

In [7]:
#write a funtcion to create the mdoel and try different parametes(kerasClassifier   )

def create_model(neurons = 32, layers=1):

    model = Sequential()
    model.add(Dense(neurons,activation='relu', input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    
    model.compile(optimizer='adam', loss ='binary_crossentropy', metrics=['accuracy'])

    return model




In [15]:
#Create a keras Classifier 
#basically it creates a model by making it compatible with scikit-learn tools
model = KerasClassifier(layers = 1, neurons=32, model = create_model, verbose=1)

In [16]:
#define the grid search parameters like key and the values that you want to give and check
param_grdi = {
    'neurons': [16, 32, 64],
    'layers': [1, 2],
    'epochs': [50, 100]
}

In [17]:
#perform grid
grid = GridSearchCV(estimator=model, param_grid=param_grdi, n_jobs=-1, cv=3,verbose=1)
grid_result = grid.fit(X_train, y_train)

print(f"Best: {grid_result.best_score_} using {grid_result.best_params_}")

Fitting 3 folds for each of 12 candidates, totalling 36 fits


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.01s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
0.00s - Debugger warning: It seems that frozen modules are being used, which may
0

Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50
Epoch 1/50


2026-04-15 03:09:46.302147: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.391261: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.656211: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.675023: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.693177: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.702593: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.
2026-04-15 03:09:46.758291: I tensorflow/core/grappler/optimizers/cust

167/167 [==============================] - 14s 26ms/step - loss: 0.6947 - accuracy: 0.6197
Epoch 2/50
167/167 [==============================] - 13s 24ms/step - loss: 0.5328 - accuracy: 0.7489
Epoch 2/50
167/167 [==============================] - 14s 26ms/step - loss: 0.6703 - accuracy: 0.6541
Epoch 2/50
  1/167 [..............................] - ETA: 4s - loss: 0.5490 - accuracy: 0.8438Epoch 2/50
Epoch 2/50
167/167 [==============================] - 14s 25ms/step - loss: 0.5854 - accuracy: 0.7028
Epoch 2/50
167/167 [==============================] - 4s 23ms/step - loss: 0.4836 - accuracy: 0.7935
Epoch 3/50
167/167 [==============================] - 4s 23ms/step - loss: 0.4348 - accuracy: 0.8060
Epoch 3/50
167/167 [==============================] - 4s 23ms/step - loss: 0.4410 - accuracy: 0.8110
Epoch 3/50
167/167 [==============================] - 4s 23ms/step - loss: 0.4418 - accuracy: 0.8091
Epoch 3/50
167/167 [==============================] - 4s 23ms/step - loss: 0.4464 - accuracy: